# Stage 4c — Consultation Package (DEC-093-R1) v1.4 — committee checked

**Purpose:** This notebook creates a **consultation package**, not final Stage 5 inferential models.

Stage 4b identified unresolved feasibility issues before Stage 5:

1. **High complete-case loss** in C-lite/C-full models (~35%–37% in prior run).
2. **Hospitalization overdispersion** in Negative Binomial diagnostics.
3. **Cox PH warning** for albumin.
4. **Early mortality may bias hospitalization counts** because hospitalization dates are unavailable.

## Clarification added in DEC-093-R1

Model B, C-minimal, C-lite, and C-full are **not the only models in the project**. They are four **hierarchical predictor sets**. Each predictor set can be evaluated across different outcomes and model families:

- 1-year mortality → logistic regression
- long-term survival → Cox regression
- hospitalization burden → count models, comparing NB and ZINB with follow-up offset

Stage 4c therefore evaluates **predictor-set feasibility and model-readiness**, not one final model.

## Hierarchical predictor sets evaluated

- **Model B:** Clinical baseline + LV_EF
- **C-minimal:** Clinical baseline + LV_EF + E/e′ septal
- **C-lite:** Clinical baseline + LV_EF + E/e′ septal + EstimatedSysPAPressure + LACavitySize
- **C-full:** Clinical baseline + LV_EF + E/e′ septal + EstimatedSysPAPressure + TR + LACavitySize

## Updated hospitalization decision logic

Hospitalization burden remains a **count/rate outcome**, not a yes/no outcome. In a dialysis cohort, binary hospitalization is expected to be clinically weak because hospitalization is common and does not capture burden.

For hospitalization counts, this notebook compares:

1. **Negative Binomial (NB)** with log follow-up offset.
2. **Zero-Inflated Negative Binomial (ZINB)** with log follow-up offset.
3. **Sensitivity analysis restricted to patients surviving at least 6 months**, to examine potential bias from early death reducing opportunity to accumulate hospitalizations.

**Important:** ZINB is a strong candidate when there are many zero hospitalization counts, but it is not declared primary automatically. The final Stage 5 choice should be based on fit, convergence, interpretability, and clinical plausibility.

**Important guardrail:** Stage 4c does not select variables based on p-values. It supports consultation about missingness, model specification, and Stage 5 readiness.


## Committee QA updates added in v1.4

The expert-review pass introduced the following corrections before using this notebook for Stage 5 decisions:

1. **NB dispersion is now calculated using the Negative Binomial variance function** (`mu + alpha * mu^2`), not a Poisson-scale denominator.
2. **ZINB count-part effect estimates are exported** for consultation, rather than only AIC/BIC fit metrics.
3. **ZINB convergence status is explicitly recorded** from the fitted model object when available.
4. **MI categorical encoding is made robust to actual observed category values**, rather than forcing hard-coded categories that may not match the dataset.
5. The output directory and metadata now match the revised checked version.


In [ ]:
# Optional: mount Google Drive in Colab
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
# Imports and optional package installation
import sys, subprocess, importlib.util, json, hashlib, warnings, math, re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from patsy import dmatrices, dmatrix

# lifelines is needed for Cox models
if importlib.util.find_spec('lifelines') is None:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines'])
    except Exception as e:
        print('WARNING: lifelines installation failed:', repr(e))

try:
    from lifelines import CoxPHFitter
    from lifelines.statistics import proportional_hazard_test
    LIFELINES_AVAILABLE = True
except Exception as e:
    print('WARNING: lifelines unavailable:', repr(e))
    LIFELINES_AVAILABLE = False

# sklearn imputation for consultation-grade MI approximation
try:
    from sklearn.experimental import enable_iterative_imputer  # noqa: F401
    from sklearn.impute import IterativeImputer
    SKLEARN_MI_AVAILABLE = True
except Exception as e:
    print('WARNING: sklearn IterativeImputer unavailable:', repr(e))
    SKLEARN_MI_AVAILABLE = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)


In [ ]:
# -----------------------
# Paths
# -----------------------
BASE_DIR = Path('/content/drive/MyDrive/dialysis')
PARAMS_DIR = BASE_DIR / 'outputs' / 'params'

STAGE1_DIR = PARAMS_DIR / 'stage1'
STAGE4_DIR = PARAMS_DIR / 'stage4'
STAGE4B_DIR = STAGE4_DIR / 'stage4b'
STAGE4_PROTOCOL_DIR = STAGE4_DIR / 'protocol'

# If your folders differ, edit these paths:
STAGE4C_OUT_DIR = PARAMS_DIR / 'stage4c_consultation_package_DEC093_R1_v1_4_committee_checked'
STAGE4C_OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MI = True
MI_M = 40  # DEC-086 default; set lower only for debugging
RANDOM_SEED = 4242

print('Stage 4c output dir:', STAGE4C_OUT_DIR)
print('lifelines available:', LIFELINES_AVAILABLE)
print('sklearn MI available:', SKLEARN_MI_AVAILABLE)

## Locate Stage 1 datasets and outcome columns

This cell is intentionally flexible because earlier stages used slightly different file names.

In [ ]:
def locate_file(directory, label, candidates, contains_all=None):
    directory = Path(directory)
    for name in candidates:
        p = directory / name
        if p.exists():
            print(f'{label}: found exact file {p.name}')
            return p
    csvs = sorted(directory.glob('*.csv'))
    if contains_all:
        hits=[]
        for p in csvs:
            lname = p.name.lower()
            if all(tok.lower() in lname for tok in contains_all):
                hits.append(p)
        if len(hits)==1:
            print(f'{label}: found keyword file {hits[0].name}')
            return hits[0]
        if len(hits)>1:
            raise FileNotFoundError(f'Multiple possible files for {label}: {[p.name for p in hits]}')
    raise FileNotFoundError(f'Could not locate {label}. Tried {candidates}. Available: {[p.name for p in csvs]}')

oneyear_path = locate_file(STAGE1_DIR, 'oneyear', [
    'stage1_oneyear_mortality.csv', 'stage1_oneyear_analysis.csv', 'one_year_analysis.csv',
    'stage1_one_year_mortality.csv', 'one_year_mortality_analysis.csv'
], contains_all=['year'])

survival_path = locate_file(STAGE1_DIR, 'survival', [
    'stage1_survival.csv', 'stage1_survival_analysis.csv', 'survival_analysis.csv',
    'stage1_full_followup_survival.csv'
], contains_all=['surv'])

hosp_path = locate_file(STAGE1_DIR, 'hospitalization', [
    'stage1_hospitalization.csv', 'stage1_hosp.csv', 'stage1_hospitalization_analysis.csv',
    'hospitalization_analysis.csv', 'stage1_hosp_analysis.csv'
], contains_all=['hosp'])

one_df = pd.read_csv(oneyear_path)
surv_df = pd.read_csv(survival_path)
hosp_df = pd.read_csv(hosp_path)

print('one-year shape:', one_df.shape)
print('survival shape:', surv_df.shape)
print('hosp shape:', hosp_df.shape)

In [ ]:
def find_col(df, exact=None, contains_all=None, contains_any=None, numeric_only=False, exclude_contains=None):
    cols=list(df.columns)
    lower={c.lower():c for c in cols}
    for e in exact or []:
        if e.lower() in lower:
            return lower[e.lower()]
    candidates=[]
    for c in cols:
        lc=c.lower()
        if exclude_contains and any(x.lower() in lc for x in exclude_contains):
            continue
        if numeric_only and not pd.api.types.is_numeric_dtype(df[c]):
            continue
        if contains_all and all(tok.lower() in lc for tok in contains_all):
            candidates.append(c)
        elif contains_any and any(tok.lower() in lc for tok in contains_any):
            candidates.append(c)
    if len(candidates)==1:
        return candidates[0]
    if len(candidates)>1:
        print('Multiple candidates:', candidates)
    return None

PID_COLS = ['patient_id','PatientID','pid','id']
pid_col = find_col(one_df, exact=PID_COLS) or find_col(surv_df, exact=PID_COLS) or find_col(hosp_df, exact=PID_COLS)
assert pid_col is not None, 'No patient identifier found.'

ONE_EVENT_COL = find_col(one_df, exact=[
    'died_1year','died_1y','one_year_mortality','mortality_1y','death_1y','died_within_1y',
    'death_within_365d','one_year_death','oneyear_mortality','one_year_mortality_event'
], contains_any=['died','death','mortality','1year','1y'])
assert ONE_EVENT_COL is not None, 'No one-year mortality event column found.'

SURV_TIME_COL = find_col(surv_df, exact=[
    'time_to_event_days','survival_time_days','survival_time','survival_days','time_to_event',
    'followup_days','follow_up_days','duration_days','days_to_death_or_censor','time_at_risk_days',
    'last_followup_days','time','t','followup_years'
], contains_any=['time_to_event','survival','followup','follow_up'], numeric_only=True,
exclude_contains=['echo','age','ef','spap','doppler','hosp','hospital','creatinine','albumin','crp','hb'])

SURV_EVENT_COL = find_col(surv_df, exact=['event','death_event','died','mortality_event','survival_event'],
                          contains_any=['event','death','died'])

HOSP_COUNT_COL = find_col(hosp_df, exact=['hosp_total','hospitalization_count','hospitalizations','n_hosp','total_hospitalizations'],
                          contains_any=['hosp_total','hospitalization','hospitalizations'])
HOSP_FU_COL = find_col(hosp_df, exact=['followup_years','follow_up_years','person_years','followup_time_years'],
                       contains_any=['followup','person_year'], numeric_only=True)

print('pid_col:', pid_col)
print('one_event_col:', ONE_EVENT_COL)
print('survival_time:', SURV_TIME_COL)
print('survival_event:', SURV_EVENT_COL)
print('hosp_count:', HOSP_COUNT_COL)
print('hosp_followup_years:', HOSP_FU_COL)

inferred_cols = pd.DataFrame([
    {'role':'patient_id','column':pid_col},
    {'role':'one_year_event','column':ONE_EVENT_COL},
    {'role':'survival_time','column':SURV_TIME_COL},
    {'role':'survival_event','column':SURV_EVENT_COL},
    {'role':'hosp_count','column':HOSP_COUNT_COL},
    {'role':'hosp_followup_years','column':HOSP_FU_COL},
])
inferred_cols.to_csv(STAGE4C_OUT_DIR / 'stage4c_inferred_columns.csv', index=False)

## Pre-specified model tiers

These tiers are based on Stage 4a/4b decisions and are not modified by Stage 4c results.

In [ ]:
CLINICAL_BASELINE = [
    'AgeAtFirstHFDate',
    'm/f',
    'albumin-numeric result',
    'creatinine-numeric result',
    'AFIB_binary',
]
EF_REF = ['LV_EF']
C_MINIMAL_EXTRA = ['TissueDopplerEERatioSeptal']
C_LITE_EXTRA = ['TissueDopplerEERatioSeptal', 'EstimatedSysPAPressure', 'LACavitySize']
C_FULL_EXTRA = ['TissueDopplerEERatioSeptal', 'EstimatedSysPAPressure', 'tricuspid_regurgitation_clin_grouped', 'LACavitySize']

MODEL_TIERS = {
    'Model_B': CLINICAL_BASELINE + EF_REF,
    'C_minimal': CLINICAL_BASELINE + EF_REF + C_MINIMAL_EXTRA,
    'C_lite': CLINICAL_BASELINE + EF_REF + C_LITE_EXTRA,
    'C_full': CLINICAL_BASELINE + EF_REF + C_FULL_EXTRA,
}

CATEGORICAL_VARS = {'m/f','AFIB_binary','LACavitySize','tricuspid_regurgitation_clin_grouped'}
CONTINUOUS_VARS = set().union(*[set(v) for v in MODEL_TIERS.values()]) - CATEGORICAL_VARS

pd.DataFrame([{'model_tier':k, 'variables':' | '.join(v), 'n_variables':len(v)} for k,v in MODEL_TIERS.items()]).to_csv(
    STAGE4C_OUT_DIR / 'stage4c_model_tier_registry.csv', index=False)
print(pd.DataFrame([{'model_tier':k, 'variables':v} for k,v in MODEL_TIERS.items()]))

## DEC-093-R1: What the four tiers mean

The four entries in `MODEL_TIERS` are **predictor sets**, not four outcomes and not the entire modeling strategy.

They are used repeatedly across outcome-specific model families:

| Predictor set | Role in Stage 4c | Expected use in Stage 5 |
|---|---|---|
| Model B | Stable baseline comparator | Mandatory reference model |
| C-minimal | Incremental echo model with limited missingness | Candidate primary interpretive model |
| C-lite | Clinically richer echo model | MI/sensitivity candidate if complete-case loss is high |
| C-full | Broadest echo model | MI/sensitivity candidate; unlikely complete-case primary if loss remains high |

This distinction avoids the incorrect interpretation that Stage 4c has only four models overall. The project has outcomes, model families, predictor sets, missingness strategies, and sensitivity analyses.


## Clinical category collapse rules

These are locked before Stage 4c and match Stage 4b logic.

In [ ]:
def normalize_str(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip()

def collapse_tr(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if any(tok in s for tok in ['moderate', 'severe', 'iii', 'iv']):
        return 'Moderate_or_Severe'
    if any(tok in s for tok in ['none', 'trace', 'trivial', 'mild', 'i']):
        return 'None_Trace_Mild'
    return str(x)

def collapse_la(x):
    if pd.isna(x): return np.nan
    s = str(x).lower().strip()
    if 'normal' in s:
        return 'Normal'
    if 'mild' in s:
        return 'Mild'
    if any(tok in s for tok in ['moderate','severe']):
        return 'Moderate_or_Severe'
    return str(x)

def collapse_echo_spap(x):
    if pd.isna(x): return np.nan
    s=str(x).lower().strip()
    if any(tok in s for tok in ['normal','mild']): return 'Normal_or_Mild'
    if any(tok in s for tok in ['moderate','severe']): return 'Moderate_or_Severe'
    return str(x)

def apply_collapses(df):
    df=df.copy()
    if 'tricuspid_regurgitation_clin_grouped' in df.columns:
        df['tricuspid_regurgitation_clin_grouped'] = df['tricuspid_regurgitation_clin_grouped'].map(collapse_tr)
    if 'LACavitySize' in df.columns:
        df['LACavitySize'] = df['LACavitySize'].map(collapse_la)
    if 'ECHO_SPAP' in df.columns:
        df['ECHO_SPAP'] = df['ECHO_SPAP'].map(collapse_echo_spap)
    return df

one_df = apply_collapses(one_df)
surv_df = apply_collapses(surv_df)
hosp_df = apply_collapses(hosp_df)

## Model-fitting helper functions

Stage 4c writes estimates for consultation. These are not final Stage 5 model outputs.

In [ ]:
def qvar(v):
    return f'Q({v!r})'

def formula_rhs(vars_):
    terms=[]
    for v in vars_:
        if v in CATEGORICAL_VARS:
            terms.append(f'C({qvar(v)})')
        else:
            terms.append(qvar(v))
    return ' + '.join(terms)

def model_df_from_design(X):
    # exclude intercept
    return max(0, X.shape[1] - (1 if 'Intercept' in X.columns else 0))

def complete_case(df, vars_, extra_cols):
    # patient_id is needed for original-data auditing, but imputed datasets used for
    # consultation-grade MI do not carry patient_id into the model matrix.
    # Therefore patient_id is included only if present, and is never required.
    required_cols = list(dict.fromkeys(vars_ + extra_cols))
    use_cols = list(required_cols)
    if pid_col in df.columns:
        use_cols.append(pid_col)
    use_cols = list(dict.fromkeys(use_cols))
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f'Missing columns: {missing}')
    d = df[use_cols].copy()
    n_total = len(d)
    d = d.dropna(subset=required_cols)
    return d, n_total

def tidy_sm_result(res, outcome, tier, model_type, effect_measure, n, events_or_counts):
    rows=[]
    params=res.params
    bse=res.bse
    for term, beta in params.items():
        if term in ['Intercept','alpha','inflate_const'] or str(term).startswith('inflate_'):
            continue
        se=bse.get(term, np.nan) if hasattr(bse,'get') else np.nan
        ci_low = beta - 1.96*se if pd.notna(se) else np.nan
        ci_high = beta + 1.96*se if pd.notna(se) else np.nan
        if effect_measure in ['OR','IRR','HR']:
            est=np.exp(beta); lo=np.exp(ci_low); hi=np.exp(ci_high)
        else:
            est=beta; lo=ci_low; hi=ci_high
        p=res.pvalues.get(term, np.nan) if hasattr(res,'pvalues') else np.nan
        rows.append({
            'outcome': outcome, 'model_tier': tier, 'model_type': model_type,
            'variable_or_term': term, 'effect_measure': effect_measure,
            'estimate': est, 'ci_lower': lo, 'ci_upper': hi, 'p_value': p,
            'n_complete': n, 'events_or_counts': events_or_counts,
            'note': 'consultation_only_not_final_stage5'
        })
    return rows

def fit_logistic(df, tier, vars_):
    d, n_total = complete_case(df, vars_, [ONE_EVENT_COL])
    y, X = dmatrices(f'{qvar(ONE_EVENT_COL)} ~ {formula_rhs(vars_)}', d, return_type='dataframe')
    res = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    events = int(d[ONE_EVENT_COL].sum())
    df_model = model_df_from_design(X)
    feas = {'outcome':'one_year_mortality','model_tier':tier,'model_type':'logistic','n_total':n_total,'n_complete':len(d),
            'n_lost':n_total-len(d),'complete_case_loss_pct':100*(n_total-len(d))/n_total,'events_or_counts':events,
            'df':df_model,'events_per_df_or_counts_per_df':events/df_model if df_model else np.nan,
            'convergence_status':'OK','warnings':''}
    return res, feas, tidy_sm_result(res,'one_year_mortality',tier,'logistic','OR',len(d),events)


def fit_cox(df, tier, vars_):
    if not LIFELINES_AVAILABLE:
        raise RuntimeError('lifelines unavailable')
    if SURV_TIME_COL is None or SURV_EVENT_COL is None:
        raise RuntimeError('survival columns missing')
    d, n_total = complete_case(df, vars_, [SURV_TIME_COL, SURV_EVENT_COL])
    design = pd.get_dummies(d[vars_], drop_first=True, dtype=float)
    cdf = pd.concat([d[[SURV_TIME_COL, SURV_EVENT_COL]].reset_index(drop=True), design.reset_index(drop=True)], axis=1)
    cph = CoxPHFitter()
    cph.fit(cdf, duration_col=SURV_TIME_COL, event_col=SURV_EVENT_COL, show_progress=False)
    events=int(d[SURV_EVENT_COL].sum())
    df_model=design.shape[1]
    rows=[]
    summ=cph.summary
    for term, r in summ.iterrows():
        rows.append({
            'outcome':'survival','model_tier':tier,'model_type':'cox','variable_or_term':term,
            'effect_measure':'HR','estimate':float(np.exp(r['coef'])),'ci_lower':float(np.exp(r['coef lower 95%'])),
            'ci_upper':float(np.exp(r['coef upper 95%'])),'p_value':float(r['p']),
            'n_complete':len(d),'events_or_counts':events,'note':'consultation_only_not_final_stage5'
        })
    feas={'outcome':'survival','model_tier':tier,'model_type':'cox','n_total':n_total,'n_complete':len(d),
          'n_lost':n_total-len(d),'complete_case_loss_pct':100*(n_total-len(d))/n_total,'events_or_counts':events,
          'df':df_model,'events_per_df_or_counts_per_df':events/df_model if df_model else np.nan,
          'convergence_status':'OK','warnings':''}

    # PH diagnostic table, saved as structured output rather than only printed text.
    ph_rows=[]
    try:
        ph = proportional_hazard_test(cph, cdf, time_transform='rank')
        ph_summary = ph.summary.reset_index().rename(columns={'index':'variable_or_term'})
        for _, r in ph_summary.iterrows():
            pval = float(r.get('p', np.nan))
            ph_rows.append({
                'model_tier': tier,
                'variable_or_term': r.get('variable_or_term', np.nan),
                'ph_test': 'lifelines_proportional_hazard_test_rank',
                'test_statistic': float(r.get('test_statistic', np.nan)),
                'p_value': pval,
                'ph_violation_p_lt_0_05': bool(pd.notna(pval) and pval < 0.05),
                'decision_needed_for_stage5': 'review_time_varying_effect_or_sensitivity' if pd.notna(pval) and pval < 0.05 else 'no_flag_from_rank_test',
                'note': 'consultation_only; confirm final PH handling with biostatistician'
            })
    except Exception as e:
        ph_rows.append({
            'model_tier': tier,
            'variable_or_term': 'PH_CHECK_ERROR',
            'ph_test': 'lifelines_proportional_hazard_test_rank',
            'test_statistic': np.nan,
            'p_value': np.nan,
            'ph_violation_p_lt_0_05': np.nan,
            'decision_needed_for_stage5': 'PH_check_failed_review_code_or_data',
            'note': repr(e)
        })
    return cph, feas, rows, ph_rows

def fit_nb(df, tier, vars_):
    if HOSP_COUNT_COL is None or HOSP_FU_COL is None:
        raise RuntimeError('hospitalization count/follow-up columns missing')
    d, n_total = complete_case(df, vars_, [HOSP_COUNT_COL, HOSP_FU_COL])
    d = d[d[HOSP_FU_COL] > 0].copy()
    y, X = dmatrices(f'{qvar(HOSP_COUNT_COL)} ~ {formula_rhs(vars_)}', d, return_type='dataframe')
    offset=np.log(d[HOSP_FU_COL].astype(float).values)
    try:
        res = sm.NegativeBinomial(y.iloc[:,0], X, offset=offset).fit(disp=False, maxiter=200)
        conv = 'OK'
    except Exception as e:
        res = sm.GLM(y, X, family=sm.families.NegativeBinomial(alpha=1.0), offset=offset).fit()
        conv = 'GLM_NB_ALPHA1_FALLBACK'
    counts=float(d[HOSP_COUNT_COL].sum())
    df_model=model_df_from_design(X)
    # Pearson dispersion from fitted mean.
    # Committee QA v1.4: use NB variance, not Poisson variance. A Poisson-scale
    # denominator can exaggerate residual overdispersion after fitting NB.
    mu=np.asarray(res.predict(X, offset=offset))
    alpha = np.nan
    try:
        if hasattr(res, 'params') and hasattr(res.params, 'get'):
            alpha = float(res.params.get('alpha', np.nan))
        if not np.isfinite(alpha):
            alpha = float(getattr(getattr(res, 'model', None), 'alpha', np.nan))
    except Exception:
        alpha = np.nan
    if not np.isfinite(alpha):
        alpha = 1.0 if conv == 'GLM_NB_ALPHA1_FALLBACK' else 0.0
    nb_var = np.maximum(mu + alpha*(mu**2), 1e-9)
    pearson=np.sum((y.iloc[:,0].values-mu)**2/nb_var) / max(1,(len(d)-X.shape[1]))
    # Also keep a Poisson-scale diagnostic for transparency/comparison only.
    pearson_poisson_scale=np.sum((y.iloc[:,0].values-mu)**2/np.maximum(mu,1e-9)) / max(1,(len(d)-X.shape[1]))
    feas={'outcome':'hospitalization','model_tier':tier,'model_type':'negative_binomial','n_total':n_total,'n_complete':len(d),
          'n_lost':n_total-len(d),'complete_case_loss_pct':100*(n_total-len(d))/n_total,'events_or_counts':counts,
          'df':df_model,'events_per_df_or_counts_per_df':counts/df_model if df_model else np.nan,
          'convergence_status':conv,
          'warnings':f'NB_Pearson_dispersion={pearson:.3f}; Poisson_scale_dispersion={pearson_poisson_scale:.3f}; alpha={alpha:.4g}',
          'pearson_dispersion':pearson,
          'pearson_dispersion_scale':'negative_binomial_variance',
          'pearson_dispersion_poisson_scale':pearson_poisson_scale,
          'nb_alpha':alpha}
    return res, feas, tidy_sm_result(res,'hospitalization',tier,'negative_binomial','IRR',len(d),counts)

def fit_zinb(df, tier, vars_):
    if HOSP_COUNT_COL is None or HOSP_FU_COL is None:
        raise RuntimeError('hospitalization count/follow-up columns missing')
    d, n_total = complete_case(df, vars_, [HOSP_COUNT_COL, HOSP_FU_COL])
    d = d[d[HOSP_FU_COL] > 0].copy()
    y, X = dmatrices(f'{qvar(HOSP_COUNT_COL)} ~ {formula_rhs(vars_)}', d, return_type='dataframe')
    offset=np.log(d[HOSP_FU_COL].astype(float).values)
    # Inflation model is intercept-only at Stage 4c to test whether excess zeros
    # improve fit. Stage 5 can revisit inflation covariates only if clinically justified.
    exog_infl = pd.DataFrame({'inflate_const': np.ones(len(d))}, index=X.index)
    res = ZeroInflatedNegativeBinomialP(y.iloc[:,0].values, X, exog_infl=exog_infl, offset=offset).fit(disp=False, maxiter=250)
    counts=float(d[HOSP_COUNT_COL].sum())
    df_model=model_df_from_design(X)+1
    converged = bool(getattr(res, 'mle_retvals', {}).get('converged', True))
    conv_status = 'OK' if converged else 'NOT_CONVERGED_REVIEW'
    feas = {'outcome':'hospitalization','model_tier':tier,'model_type':'ZINB','n_total':n_total,'n_complete':len(d),
            'n_lost':n_total-len(d),'complete_case_loss_pct':100*(n_total-len(d))/n_total,'events_or_counts':counts,
            'df':df_model,'events_per_df_or_counts_per_df':counts/df_model if df_model else np.nan,
            'convergence_status':conv_status,
            'warnings':'inflation_model_intercept_only; review_ZINB_clinical_plausibility'}
    rows = tidy_sm_result(res,'hospitalization',tier,'ZINB_count_part','IRR',len(d),counts)
    for r in rows:
        r['note'] = 'consultation_only_not_final_stage5; ZINB count part only; inflation intercept not interpreted clinically'
    return res, feas, rows


## Run complete-case consultation models

In [ ]:
feasibility=[]
estimates=[]
cox_ph_rows=[]
hosp_model_compare=[]

for tier, vars_ in MODEL_TIERS.items():
    # 1-year logistic
    try:
        res, feas, rows = fit_logistic(one_df, tier, vars_)
        feasibility.append(feas); estimates.extend(rows)
    except Exception as e:
        feasibility.append({'outcome':'one_year_mortality','model_tier':tier,'model_type':'logistic','convergence_status':'ERROR','warnings':repr(e)})

    # survival Cox
    try:
        cph, feas, rows, ph_rows = fit_cox(surv_df, tier, vars_)
        feasibility.append(feas); estimates.extend(rows)
        cox_ph_rows.extend(ph_rows)
    except Exception as e:
        feasibility.append({'outcome':'survival','model_tier':tier,'model_type':'cox','convergence_status':'ERROR','warnings':repr(e)})
        cox_ph_rows.append({'model_tier':tier, 'variable_or_term':'COX_MODEL_ERROR', 'ph_test':'not_run', 'test_statistic':np.nan, 'p_value':np.nan, 'ph_violation_p_lt_0_05':np.nan, 'decision_needed_for_stage5':'cox_model_failed', 'note':repr(e)})

    # hospitalization NB
    try:
        res, feas, rows = fit_nb(hosp_df, tier, vars_)
        feasibility.append(feas); estimates.extend(rows)
        hosp_model_compare.append({**feas, 'AIC': getattr(res,'aic',np.nan), 'BIC': getattr(res,'bic',np.nan), 'count_model_type':'NB'})
    except Exception as e:
        feasibility.append({'outcome':'hospitalization','model_tier':tier,'model_type':'negative_binomial','convergence_status':'ERROR','warnings':repr(e)})

    # hospitalization ZINB sensitivity
    try:
        zres, zfeas, zrows = fit_zinb(hosp_df, tier, vars_)
        estimates.extend(zrows)
        hosp_model_compare.append({**zfeas, 'AIC': getattr(zres,'aic',np.nan), 'BIC': getattr(zres,'bic',np.nan), 'count_model_type':'ZINB'})
    except Exception as e:
        hosp_model_compare.append({'outcome':'hospitalization','model_tier':tier,'count_model_type':'ZINB','convergence_status':'ERROR','warnings':repr(e)})

feas_df=pd.DataFrame(feasibility)
est_df=pd.DataFrame(estimates)
ph_df=pd.DataFrame(cox_ph_rows)
hosp_comp_df=pd.DataFrame(hosp_model_compare)

feas_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_model_feasibility_summary.csv', index=False)
est_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_completecase_estimates.csv', index=False)
ph_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_cox_ph_consultation_status.csv', index=False)
hosp_comp_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_hospitalization_model_comparison.csv', index=False)

display(feas_df)
print('Cox PH structured diagnostic:')
display(ph_df)


## Missingness impact summary

In [ ]:
def missingness_impact(df, vars_, label):
    out=[]
    n=len(df)
    for v in vars_:
        if v in df.columns:
            out.append({'model_tier':label,'variable':v,'n_missing':int(df[v].isna().sum()),'missing_pct':100*df[v].isna().mean()})
        else:
            out.append({'model_tier':label,'variable':v,'n_missing':np.nan,'missing_pct':np.nan,'note':'missing column'})
    # complete-case loss
    cc=df[vars_].dropna().shape[0] if all(v in df.columns for v in vars_) else np.nan
    for r in out:
        r['n_total']=n; r['n_complete_predictors_only']=cc; r['predictor_completecase_loss_pct']=100*(n-cc)/n if pd.notna(cc) else np.nan
    return out

miss=[]
for tier, vars_ in MODEL_TIERS.items():
    miss.extend(missingness_impact(one_df, vars_, tier + '_oneyear'))
    miss.extend(missingness_impact(surv_df, vars_, tier + '_survival'))
    miss.extend(missingness_impact(hosp_df, vars_, tier + '_hosp'))
miss_df=pd.DataFrame(miss)
miss_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_missingness_impact.csv', index=False)
display(miss_df.sort_values('missing_pct', ascending=False).head(20))

## Candidate substitutions due to missingness

This table addresses a key consultation question: when C-lite/C-full lose many patients due to missing echo measurements, are there clinically adjacent lower-missingness representatives within the same echo domain?

These are **not automatic substitutions**. They are prompts for biostatistician/nephrologist/cardiologist review and must not be chosen based on effect size or p-values.

In [ ]:

def variable_missing_pct_by_dataset(var):
    rows=[]
    for outcome, df in [('one_year_mortality', one_df), ('survival', surv_df), ('hospitalization', hosp_df)]:
        if var in df.columns:
            rows.append({
                'outcome': outcome,
                'variable': var,
                'n_total': len(df),
                'n_missing': int(df[var].isna().sum()),
                'missing_pct': 100*df[var].isna().mean()
            })
        else:
            rows.append({
                'outcome': outcome,
                'variable': var,
                'n_total': len(df),
                'n_missing': np.nan,
                'missing_pct': np.nan,
                'note': 'variable_not_available_in_dataset'
            })
    return pd.DataFrame(rows)

def max_missing_pct(var):
    tmp = variable_missing_pct_by_dataset(var)
    return tmp['missing_pct'].max(skipna=True)

def summarize_model_impact(var):
    tiers = [tier for tier, vars_ in MODEL_TIERS.items() if var in vars_]
    return '; '.join(tiers) if tiers else 'not_in_primary_tiers'

substitution_plan = [
    {
        'clinical_domain': 'Diastolic / filling pressure',
        'current_primary_variable': 'TissueDopplerEERatioSeptal',
        'current_role': 'primary non-EF echo marker in C-minimal/C-lite/C-full',
        'candidate_substitutes': [
            ('TissueDopplerEERatioLateral', 'adjacent E/e′ marker; not identical to septal E/e′'),
            ('TissueDopplerEVelositySeptal', 'adjacent tissue Doppler e′ velocity; related but not same construct'),
            ('TissueDopplerEVelosityLateral', 'adjacent tissue Doppler e′ velocity; related but not same construct'),
            ('MitralInflowPeakEWave', 'hemodynamic/filling-adjacent marker; previously considered hospitalization-specific discovery'),
            ('LACavitySize', 'chronic filling burden marker; not a direct substitute for E/e′')
        ],
        'committee_question': 'If septal E/e′ drives loss, is any adjacent diastolic marker acceptable, or should E/e′ remain the only primary filling-pressure representative?'
    },
    {
        'clinical_domain': 'Pulmonary pressure',
        'current_primary_variable': 'EstimatedSysPAPressure',
        'current_role': 'primary pulmonary pressure representative in C-lite/C-full',
        'candidate_substitutes': [
            ('ECHO_SPAP', 'categorical pulmonary pressure alternative; may reduce interpretability/df depending on missingness'),
            ('tricuspid_regurgitation_clin_grouped', 'right-sided burden/valvular adjacent marker; not direct pressure measurement')
        ],
        'committee_question': 'If SPAP drives loss, is ECHO_SPAP an acceptable pressure substitute, or should pulmonary pressure remain MI/sensitivity only?'
    },
    {
        'clinical_domain': 'Right-sided / valvular burden',
        'current_primary_variable': 'tricuspid_regurgitation_clin_grouped',
        'current_role': 'included only in C-full; removed from locked C-lite due to SPAP overlap/df',
        'candidate_substitutes': [
            ('EstimatedSysPAPressure', 'pulmonary pressure representative; C-lite already uses this instead of TR'),
            ('ECHO_SPAP', 'categorical pressure alternative'),
            ('omit_TR_keep_SPAP', 'not a data column: use SPAP-only as C-lite strategy')
        ],
        'committee_question': 'Should right-sided burden be represented by SPAP only in the primary model, leaving TR for sensitivity/MI models?'
    },
    {
        'clinical_domain': 'LA / chronic filling burden',
        'current_primary_variable': 'LACavitySize',
        'current_role': 'chronic burden marker retained in C-lite/C-full',
        'candidate_substitutes': [
            ('TissueDopplerEERatioSeptal', 'acute/estimated filling pressure; not chronic structural burden'),
            ('MitralInflowPeakEWave', 'hemodynamic/filling-adjacent marker; not structural LA burden'),
            ('omit_LA_keep_EEprime_SPAP', 'not a data column: drop LA size if loss/df unacceptable')
        ],
        'committee_question': 'Does LA size provide enough chronic-burden information to justify missingness, or should it move to MI/sensitivity?'
    },
]

sub_rows=[]
for block in substitution_plan:
    cur = block['current_primary_variable']
    cur_missing = max_missing_pct(cur) if cur in set(one_df.columns).union(surv_df.columns).union(hosp_df.columns) else np.nan
    for sub, interp in block['candidate_substitutes']:
        is_data_col = sub in set(one_df.columns).union(surv_df.columns).union(hosp_df.columns)
        sub_missing = max_missing_pct(sub) if is_data_col else np.nan
        if not is_data_col and sub.startswith('omit_'):
            rec_use = 'model simplification strategy; not a variable substitute'
            domain_fit = 'removes domain rather than substituting a measured marker'
            true_sub = 'no'
            adjacent = 'strategy'
        else:
            rec_use = 'committee_review_only; do_not_auto_replace'
            domain_fit = interp
            true_sub = 'partial' if 'not' in interp.lower() or 'adjacent' in interp.lower() else 'yes'
            adjacent = 'yes' if true_sub in ['partial','yes'] else 'no'
        sub_rows.append({
            'clinical_domain': block['clinical_domain'],
            'current_primary_variable': cur,
            'current_role': block['current_role'],
            'current_missing_pct_max_across_outcomes': cur_missing,
            'current_model_impact': summarize_model_impact(cur),
            'candidate_substitute': sub,
            'candidate_is_data_column': bool(is_data_col),
            'substitute_missing_pct_max_across_outcomes': sub_missing,
            'substitute_domain_fit': domain_fit,
            'is_true_substitute': true_sub,
            'is_adjacent_marker_or_strategy': adjacent,
            'recommended_use': rec_use,
            'committee_question': block['committee_question'],
            'guardrail': 'Substitution must be based on clinical-domain fit and missingness, not on effect size or p-value.'
        })

sub_df = pd.DataFrame(sub_rows)
sub_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_candidate_substitution_due_to_missingness.csv', index=False)

# Long missingness table for current variables and candidate substitutes.
vars_for_sub_audit = sorted(set(sub_df['current_primary_variable']).union(
    {v for v in sub_df['candidate_substitute'] if v in set(one_df.columns).union(surv_df.columns).union(hosp_df.columns)}
))
miss_long = pd.concat([variable_missing_pct_by_dataset(v) for v in vars_for_sub_audit], ignore_index=True)
miss_long.to_csv(STAGE4C_OUT_DIR / 'stage4c_candidate_substitution_missingness_long.csv', index=False)

display(sub_df)


## Hospitalization zero/outlier diagnostics

In [ ]:
if HOSP_COUNT_COL is not None:
    h = hosp_df[HOSP_COUNT_COL].dropna().astype(float)
    zero_diag = pd.DataFrame([{
        'n_patients': len(h),
        'pct_zero_hospitalizations': 100*(h==0).mean(),
        'mean_hospitalizations': h.mean(),
        'median_hospitalizations': h.median(),
        'iqr_hospitalizations': f'{h.quantile(0.25):.1f}-{h.quantile(0.75):.1f}',
        'max_hospitalizations': h.max(),
        'top_1pct_hospitalization_share': h.sort_values(ascending=False).head(max(1,int(np.ceil(0.01*len(h))))).sum()/max(1,h.sum()),
        'comment':'Consultation diagnostic for NB overdispersion / zero inflation'
    }])
else:
    zero_diag = pd.DataFrame([{'comment':'HOSP_COUNT_COL not found'}])
zero_diag.to_csv(STAGE4C_OUT_DIR / 'stage4c_hospitalization_zero_outlier_diagnostics.csv', index=False)
display(zero_diag)

## DEC-094: Early mortality sensitivity for hospitalization burden

Because hospitalization data are available as total counts without dates, early death can reduce the opportunity to accumulate hospitalizations. This is a real limitation, not a reason to convert hospitalization burden to a binary yes/no outcome.

This section creates a sensitivity analysis restricted to patients with at least 6 months of observed survival time. It is intended to answer whether the NB/ZINB hospitalization findings are qualitatively stable after excluding patients with very early death or very short observed survival.


In [ ]:
def six_month_threshold_for_survival_time():
    """Infer threshold from survival-time column name. Earlier Stage notebooks usually use days."""
    if SURV_TIME_COL is None:
        return np.nan, 'unknown'
    lc = str(SURV_TIME_COL).lower()
    if 'year' in lc:
        return 0.5, 'years'
    if 'month' in lc:
        return 6.0, 'months'
    return 182.625, 'days'

def add_survival_info_to_hosp():
    if SURV_TIME_COL is None or SURV_EVENT_COL is None or pid_col is None:
        return None, 'missing survival columns or patient id'
    needed = [pid_col, SURV_TIME_COL, SURV_EVENT_COL]
    missing = [c for c in needed if c not in surv_df.columns]
    if missing:
        return None, f'missing in survival df: {missing}'
    h = hosp_df.copy()
    if SURV_TIME_COL in h.columns and SURV_EVENT_COL in h.columns:
        return h, 'hospitalization df already contains survival columns'
    s = surv_df[needed].drop_duplicates(subset=[pid_col]).copy()
    merged = h.merge(s, on=pid_col, how='left', suffixes=('', '_surv'))
    return merged, 'merged survival columns into hospitalization df'

def fit_hospitalization_count_models_for_dataset(df_in, cohort_label):
    rows=[]
    for tier, vars_ in MODEL_TIERS.items():
        try:
            nb_res, nb_feas, nb_rows = fit_nb(df_in, tier, vars_)
            rows.append({**nb_feas, 'AIC': getattr(nb_res,'aic',np.nan), 'BIC': getattr(nb_res,'bic',np.nan),
                         'count_model_type':'NB', 'cohort_label':cohort_label})
        except Exception as e:
            rows.append({'outcome':'hospitalization','model_tier':tier,'count_model_type':'NB',
                         'cohort_label':cohort_label,'convergence_status':'ERROR','warnings':repr(e)})
        try:
            zinb_res, zinb_feas, zinb_rows = fit_zinb(df_in, tier, vars_)
            rows.append({**zinb_feas, 'AIC': getattr(zinb_res,'aic',np.nan), 'BIC': getattr(zinb_res,'bic',np.nan),
                         'count_model_type':'ZINB', 'cohort_label':cohort_label})
        except Exception as e:
            rows.append({'outcome':'hospitalization','model_tier':tier,'count_model_type':'ZINB',
                         'cohort_label':cohort_label,'convergence_status':'ERROR','warnings':repr(e)})
    return pd.DataFrame(rows)

hosp_surv_df, merge_note = add_survival_info_to_hosp()
threshold_6m, time_unit = six_month_threshold_for_survival_time()

if hosp_surv_df is None or SURV_TIME_COL not in hosp_surv_df.columns:
    early_mortality_desc = pd.DataFrame([{'status':'six_month_sensitivity_unavailable','note':merge_note}])
    hosp_6m_compare_df = pd.DataFrame([{'status':'six_month_sensitivity_unavailable','note':merge_note}])
else:
    observed_6m_mask = pd.to_numeric(hosp_surv_df[SURV_TIME_COL], errors='coerce') >= threshold_6m
    early_death_mask = (pd.to_numeric(hosp_surv_df[SURV_TIME_COL], errors='coerce') < threshold_6m) & (hosp_surv_df[SURV_EVENT_COL].astype(float) == 1)
    short_observation_mask = pd.to_numeric(hosp_surv_df[SURV_TIME_COL], errors='coerce') < threshold_6m
    hosp_6m_df = hosp_surv_df.loc[observed_6m_mask].copy()

    early_mortality_desc = pd.DataFrame([{
        'n_total_hospitalization_dataset': len(hosp_surv_df),
        'survival_time_column': SURV_TIME_COL,
        'survival_time_unit_inferred': time_unit,
        'six_month_threshold_used': threshold_6m,
        'n_with_observed_survival_at_least_6m': int(observed_6m_mask.sum()),
        'pct_with_observed_survival_at_least_6m': 100*observed_6m_mask.mean(),
        'n_death_before_6m': int(early_death_mask.sum()),
        'pct_death_before_6m': 100*early_death_mask.mean(),
        'n_survival_time_less_than_6m_any_status': int(short_observation_mask.sum()),
        'pct_survival_time_less_than_6m_any_status': 100*short_observation_mask.mean(),
        'note': '6-month survivor sensitivity for hospitalization burden; exclude time<6m from sensitivity cohort'
    }])
    hosp_6m_compare_df = fit_hospitalization_count_models_for_dataset(hosp_6m_df, 'survived_or_observed_at_least_6_months')

early_mortality_desc.to_csv(STAGE4C_OUT_DIR / 'stage4c_early_mortality_6m_description.csv', index=False)
hosp_6m_compare_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_hospitalization_NB_ZINB_6m_survivor_sensitivity.csv', index=False)

display(early_mortality_desc)
display(hosp_6m_compare_df)


## Consultation-grade multiple imputation comparison

This section is intentionally labeled as consultation-grade. It supports discussion about whether MI materially changes conclusions. Final MI implementation should be confirmed by a biostatistician before Stage 5.

Default: C-lite and C-full only, because these tiers triggered high complete-case loss in Stage 4b.

In [ ]:
MI_TIERS = ['C_minimal','C_lite','C_full']  # DEC-093-R1: include C-minimal as optional MI sensitivity as well

# Preferred category order where it matches actual data. v1.4 guardrail:
# do not force hard-coded categories if the dataset uses different values.
PREFERRED_CATEGORY_LEVELS = {
    'm/f': ['m','f'],
    'AFIB_binary': [0,1],
    'LACavitySize': ['Normal','Mild','Moderate_or_Severe'],
    'tricuspid_regurgitation_clin_grouped': ['None_Trace_Mild','Moderate_or_Severe'],
}

def observed_category_levels(series, preferred=None):
    obs = [x for x in pd.Series(series).dropna().unique().tolist()]
    if preferred:
        levels = [x for x in preferred if x in obs]
        levels += [x for x in obs if x not in levels]
        return levels if levels else obs
    try:
        return sorted(obs)
    except Exception:
        return obs

def encode_for_imputation(df, vars_, outcome_cols):
    d = df[list(dict.fromkeys(vars_ + outcome_cols))].copy()
    enc = d.copy()
    cat_maps={}
    for v in vars_:
        if v in CATEGORICAL_VARS and v in enc.columns:
            observed = observed_category_levels(enc[v], PREFERRED_CATEGORY_LEVELS.get(v))
            if not observed:
                cat_maps[v]=[]
                enc[v]=np.nan
                continue
            cat_maps[v]=observed
            enc[v] = pd.Categorical(enc[v], categories=observed).codes.astype(float)
            # pandas uses -1 for values outside categories; treat those as missing.
            enc.loc[(enc[v] < 0) | d[v].isna(), v] = np.nan
        elif v in enc.columns:
            enc[v] = pd.to_numeric(enc[v], errors='coerce')
    return enc, cat_maps

def decode_imputed(enc, cat_maps):
    d=enc.copy()
    for v, levels in cat_maps.items():
        if v in d.columns:
            vals=np.rint(d[v]).astype(int)
            vals=np.clip(vals, 0, len(levels)-1)
            d[v]=[levels[i] for i in vals]
    return d

def fit_on_imputed_once(df_imp, outcome, tier, vars_):
    if outcome=='one_year_mortality':
        res, feas, rows = fit_logistic(df_imp, tier, vars_)
        return rows
    if outcome=='survival':
        cph, feas, rows, ph_rows = fit_cox(df_imp, tier, vars_)
        return rows
    if outcome=='hospitalization':
        # MI pooling is limited to NB in Stage 4c. ZINB MI is deferred to Stage 5
        # if ZINB is selected, because pooling mixture-model parameters requires
        # additional biostatistical decisions.
        res, feas, rows = fit_nb(df_imp, tier, vars_)
        return rows
    return []

def mi_compare_for(outcome, df, tier, vars_, outcome_cols):
    if not (RUN_MI and SKLEARN_MI_AVAILABLE):
        return pd.DataFrame([{'outcome':outcome,'model_tier':tier,'status':'MI_SKIPPED'}])
    enc, cat_maps = encode_for_imputation(df, vars_, outcome_cols)
    # Keep only rows with observed outcomes/follow-up; impute predictors only.
    enc = enc.dropna(subset=outcome_cols).copy()
    pooled=[]
    all_rows=[]
    for m in range(MI_M):
        imp = IterativeImputer(random_state=RANDOM_SEED+m, sample_posterior=True, max_iter=20)
        arr = imp.fit_transform(enc)
        imp_df = pd.DataFrame(arr, columns=enc.columns)
        # restore outcomes exactly from original observed values
        for oc in outcome_cols:
            imp_df[oc] = enc[oc].values
        imp_dec = decode_imputed(imp_df, cat_maps)
        # For binary outcomes/events, restore integer
        for oc in outcome_cols:
            if oc in [ONE_EVENT_COL, SURV_EVENT_COL]:
                imp_dec[oc] = imp_dec[oc].round().astype(int)
        try:
            rows = fit_on_imputed_once(imp_dec, outcome, tier, vars_)
            for r in rows:
                r['imputation'] = m+1
            all_rows.extend(rows)
        except Exception as e:
            all_rows.append({'outcome':outcome,'model_tier':tier,'imputation':m+1,'status':'ERROR','error':repr(e)})
    miraw=pd.DataFrame(all_rows)
    if miraw.empty or 'estimate' not in miraw.columns:
        return miraw
    # Pool on log(effect) scale approximately using mean and between-imputation SD; consultation only.
    good = miraw.dropna(subset=['estimate']).copy()
    good['log_estimate'] = np.log(good['estimate'].astype(float))
    grp = good.groupby(['outcome','model_tier','model_type','variable_or_term','effect_measure'], dropna=False)
    for keys, g in grp:
        qbar = g['log_estimate'].mean()
        b = g['log_estimate'].var(ddof=1) if len(g)>1 else 0
        # Within-imputation variance unavailable from saved rows; approximate using CI width when present.
        if {'ci_lower','ci_upper'}.issubset(g.columns):
            se_w = ((np.log(g['ci_upper']) - np.log(g['ci_lower']))/(2*1.96))**2
            w = se_w.mean()
        else:
            w = np.nan
        tvar = w + (1+1/MI_M)*b if pd.notna(w) else b
        se = math.sqrt(max(tvar,0)) if pd.notna(tvar) else np.nan
        pooled.append({
            'outcome':keys[0],'model_tier':keys[1],'model_type':keys[2],'variable_or_term':keys[3],
            'effect_measure':keys[4], 'estimate_MI':np.exp(qbar),
            'ci_lower_MI':np.exp(qbar-1.96*se) if pd.notna(se) else np.nan,
            'ci_upper_MI':np.exp(qbar+1.96*se) if pd.notna(se) else np.nan,
            'MI_M':MI_M, 'MI_note':'consultation_grade_approximation_confirm_with_biostatistician'
        })
    return pd.DataFrame(pooled)

mi_tables=[]
for tier in MI_TIERS:
    vars_=MODEL_TIERS[tier]
    mi_tables.append(mi_compare_for('one_year_mortality', one_df, tier, vars_, [ONE_EVENT_COL]))
    if SURV_TIME_COL and SURV_EVENT_COL and LIFELINES_AVAILABLE:
        mi_tables.append(mi_compare_for('survival', surv_df, tier, vars_, [SURV_TIME_COL, SURV_EVENT_COL]))
    if HOSP_COUNT_COL and HOSP_FU_COL:
        mi_tables.append(mi_compare_for('hospitalization', hosp_df, tier, vars_, [HOSP_COUNT_COL, HOSP_FU_COL]))

mi_df = pd.concat(mi_tables, ignore_index=True) if mi_tables else pd.DataFrame()
mi_df.to_csv(STAGE4C_OUT_DIR / 'stage4c_MI_pooled_estimates_consultation.csv', index=False)

# Compare complete-case and MI on matching terms
if not mi_df.empty and not est_df.empty and 'estimate_MI' in mi_df.columns:
    comp = est_df.merge(mi_df, on=['outcome','model_tier','model_type','variable_or_term','effect_measure'], how='inner')
    comp['relative_change_pct'] = 100*np.abs(np.log(comp['estimate_MI']) - np.log(comp['estimate'])) / np.maximum(np.abs(np.log(comp['estimate'])), 1e-9)
    comp['ci_overlap'] = ~((comp['ci_upper'] < comp['ci_lower_MI']) | (comp['ci_upper_MI'] < comp['ci_lower']))
    comp['robustness_status'] = np.where((comp['relative_change_pct'] < 20) & (comp['ci_overlap']), 'robust_by_DEC077', 'not_robust_review')
else:
    comp = pd.DataFrame([{'status':'MI comparison unavailable'}])
comp.to_csv(STAGE4C_OUT_DIR / 'stage4c_completecase_vs_MI_comparison.csv', index=False)
display(comp.head(20))


## Consultation brief

In [ ]:
# Summarize headline issues into a brief markdown report
now=datetime.now().isoformat(timespec='seconds')
feas_path = STAGE4C_OUT_DIR / 'stage4c_model_feasibility_summary.csv'
est_path = STAGE4C_OUT_DIR / 'stage4c_completecase_estimates.csv'
mi_path = STAGE4C_OUT_DIR / 'stage4c_completecase_vs_MI_comparison.csv'
hosp_path = STAGE4C_OUT_DIR / 'stage4c_hospitalization_model_comparison.csv'
zero_path = STAGE4C_OUT_DIR / 'stage4c_hospitalization_zero_outlier_diagnostics.csv'
sub_path = STAGE4C_OUT_DIR / 'stage4c_candidate_substitution_due_to_missingness.csv'
ph_path = STAGE4C_OUT_DIR / 'stage4c_cox_ph_consultation_status.csv'
early_path = STAGE4C_OUT_DIR / 'stage4c_early_mortality_6m_description.csv'
hosp6m_path = STAGE4C_OUT_DIR / 'stage4c_hospitalization_NB_ZINB_6m_survivor_sensitivity.csv'

# extract tier summary loss if available
loss_summary=''
try:
    tmp=feas_df[['outcome','model_tier','model_type','n_complete','complete_case_loss_pct','df','events_per_df_or_counts_per_df','convergence_status','warnings']].copy()
    loss_summary = tmp.to_markdown(index=False)
except Exception:
    loss_summary='Feasibility summary unavailable.'

brief = f"""
# Stage 4c Consultation Brief — DEC-093-R1 v1.4 Committee-Checked

Generated: {now}

## Status

These analyses are **not final Stage 5 models**. They were generated after Stage 4b identified unresolved feasibility issues and are intended for consultation with a biostatistician and nephrologist.

## Study aim

Evaluate whether echocardiographic parameters around dialysis initiation improve risk stratification for mortality and hospitalization burden beyond LV_EF alone.

## Model tiers evaluated

These are hierarchical **predictor sets**, not the entire set of project models. They are evaluated across outcome-specific model families and missingness/sensitivity strategies.

- Model B: clinical baseline + LV_EF
- C-minimal: Model B + E/e′ septal
- C-lite: Model B + E/e′ septal + EstimatedSysPAPressure + LACavitySize
- C-full: Model B + E/e′ septal + EstimatedSysPAPressure + TR + LACavitySize

## Key questions for consultation

1. Should C-minimal complete-case be the primary model, with C-lite/C-full as mandatory MI sensitivity?
2. Or should C-lite/C-full with MI become primary due to high complete-case loss?
3. For hospitalization burden, should NB or ZINB be the Stage 5 primary count model? This must be decided empirically using model fit, convergence, interpretability, and clinical plausibility.
4. Does the ≥6-month survivor sensitivity analysis materially change hospitalization findings?
5. How should PH warnings in Cox be handled in Stage 5?
6. Should C-minimal be the primary interpretive model with C-lite/C-full as MI/sensitivity models?
7. Does C-minimal preserve enough clinical meaning for the question of incremental value beyond EF?
8. If C-lite/C-full lose too many patients, are any lower-missingness substitutes clinically acceptable within the same echo domain?

## Feasibility summary

{loss_summary}

## Output files

- `{feas_path.name}`
- `{est_path.name}`
- `{mi_path.name}`
- `{hosp_path.name}`
- `{zero_path.name}`
- `stage4c_missingness_impact.csv`
- `stage4c_candidate_substitution_due_to_missingness.csv`
- `stage4c_candidate_substitution_missingness_long.csv`
- `stage4c_cox_ph_consultation_status.csv`
- `stage4c_early_mortality_6m_description.csv`
- `stage4c_hospitalization_NB_ZINB_6m_survivor_sensitivity.csv`

## Interpretation guardrail

Do not choose the final Stage 5 primary model solely because a p-value or estimate looks favorable. Stage 4c is for consultation regarding missingness, robustness, PH diagnostics, early-mortality sensitivity, and count-model specification.

## Draft limitations language for hospitalization outcome

Hospitalization data were available as total counts without dates of individual admissions. Therefore, recurrent-event models or competing-risk recurrent-event approaches could not be applied. Although hospitalization rates were modeled using follow-up time as an offset, early mortality may have reduced the opportunity to observe hospitalizations and may therefore lead to underestimation of hospitalization burden among the sickest patients. A sensitivity analysis restricted to patients with at least 6 months of observed survival time was therefore added for Stage 5 planning.
"""

brief_path = STAGE4C_OUT_DIR / 'stage4c_consultation_brief.md'
brief_path.write_text(brief, encoding='utf-8')
print(brief_path)
print(brief[:2000])


In [ ]:
# Metadata
metadata = {
    'stage': 'Stage 4c Consultation Package',
    'version': 'DEC093_R1_v1_4_committee_checked',
    'run_timestamp': datetime.now().isoformat(timespec='seconds'),
    'purpose': 'Consultation package, not final Stage 5 model fitting',
    'oneyear_path': str(oneyear_path),
    'survival_path': str(survival_path),
    'hospitalization_path': str(hosp_path),
    'inferred_columns': inferred_cols.to_dict(orient='records'),
    'run_mi': RUN_MI,
    'mi_m': MI_M,
    'lifelines_available': LIFELINES_AVAILABLE,
    'sklearn_mi_available': SKLEARN_MI_AVAILABLE,
    'outputs_dir': str(STAGE4C_OUT_DIR),
    'guardrail': 'Not final Stage 5; predictor tiers are hierarchical predictor sets; no new variable selection; hospitalization remains count/rate outcome; NB and ZINB compared empirically; NB dispersion uses NB variance; ZINB count-part estimates exported; 6-month survivor sensitivity added; substitution table is for clinical-domain/missingness consultation only.'
}
(STAGE4C_OUT_DIR / 'stage4c_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Stage 4c consultation package complete.')
print('Outputs saved to:', STAGE4C_OUT_DIR)
